# 04 — Data Quality Checks — ShipTrack Week 6

**Team:** 10 | **Project:** ShipTrack: Supply Chain Visibility Hub
**Input:** Silver tables from `03_silver_transformations.ipynb` | **Output:** one Trusted and one Quarantine Delta table per entity

| Part | Entity | Tables written |
|---|---|---|
| A | Shipments | `candidate_shipments`, `dq_shipments_eval`, `trusted_shipments`, `quarantine_shipments` |
| B | Hubs, Carriers, Routes, Scan events, Exceptions | `candidate_*`, `dq_*_eval`, `trusted_*`, `quarantine_*` for each entity |

Outputs below are from the team's Databricks runs. Result tables show at most 20 rows.

---
# Part A — Shipments

# 06 — DQ Framework: Shipments (Candidate to Trusted / Quarantine)

**Week:** 6  
**Entity:** Shipments (first entity — extensible to hubs, carriers, routes, scans, exceptions)

## Flow

```
silver_shipments
      |
candidate_shipments
      |
      +------------------+
      |                  |
trusted_shipments   quarantine_shipments
```

## What this notebook does

1. Copies `silver_shipments` into `candidate_shipments` (the input to DQ checks)
2. Evaluates all 8 Team 10 DQ rules against each candidate row
3. Routes rows that pass ALL rules to `trusted_shipments`
4. Routes rows that fail ANY rule to `quarantine_shipments` (one row per source_record_id, with all failed rule IDs in an array)
5. Runs reconciliation checks to prove no records are lost or duplicated

**Important:** This notebook does NOT modify `silver_shipments`. It creates new tables: `candidate_shipments`, `dq_shipments_eval`, `trusted_shipments`, and `quarantine_shipments`.

## Step 0 — Schema of silver_shipments (27 columns)

| # | Column | Type |
|---|--------|------|
| 0 | source_record_id | STRING |
| 1 | shipment_id | STRING |
| 2 | order_reference | STRING |
| 3 | route_id | STRING |
| 4 | origin_hub_id | STRING |
| 5 | destination_hub_id | STRING |
| 6 | carrier_id | STRING |
| 7 | service_level | STRING |
| 8 | transport_mode | STRING |
| 9 | booking_ts | TIMESTAMP |
| 10 | pickup_ts | TIMESTAMP |
| 11 | promised_delivery_ts | TIMESTAMP |
| 12 | actual_delivery_ts | TIMESTAMP |
| 13 | return_completed_ts | TIMESTAMP |
| 14 | latest_status | STRING |
| 15 | delivery_outcome | STRING |
| 16 | status_reason | STRING |
| 17 | package_weight_kg | DOUBLE |
| 18 | package_count | BIGINT |
| 19 | freight_amount_inr | DOUBLE |
| 20 | attempt_count | BIGINT |
| 21 | exception_flag | BOOLEAN |
| 22 | source_system | STRING |
| 23 | run_id | STRING |
| 24 | ingestion_timestamp | TIMESTAMP |
| 25 | source_file_name | STRING |
| 26 | batch_id | STRING |

## Reference tables used by DQ rules

* `silver_hubs` — for DQ-REF-001 (origin/destination hub existence)
* `silver_carriers` — for DQ-REF-001 (carrier existence)
* `silver_routes` — for DQ-REF-001 (route existence) and DQ-RTE-001 (route consistency)
* `silver_scan_events` — for DQ-SCN-001 (scan sequence check; join: `silver_scan_events.reference_id` = `silver_shipments.shipment_id`)

## DQ Rule Mapping — which columns and tables each rule uses

| Rule ID | Rule Name | Columns Used | Other Tables? | What it checks |
|---------|-----------|-------------|---------------|----------------|
| DQ-SHP-001 | Shipment identity | source_record_id, shipment_id, order_reference | No | Required identity fields are not NULL; shipment_id is unique |
| DQ-REF-001 | Reference integrity | origin_hub_id, destination_hub_id, carrier_id, route_id | silver_hubs, silver_carriers, silver_routes | Referenced IDs exist in their reference tables |
| DQ-TIM-001 | Chronology | booking_ts, pickup_ts, actual_delivery_ts, return_completed_ts | No | Timestamps follow logical business order |
| DQ-SCN-001 | Scan sequence | shipment_id (links to silver_scan_events.reference_id) | silver_scan_events | Linked scan events have valid sequence numbers (>= 1, not NULL) |
| DQ-RTE-001 | Route consistency | route_id, origin_hub_id, destination_hub_id | silver_routes | Shipment hubs match the route's defined hubs |
| DQ-DEL-001 | Delivery/status consistency | latest_status, delivery_outcome, actual_delivery_ts, return_completed_ts | No | Status and delivery fields are logically consistent |
| DQ-MEA-001 | Measures | package_weight_kg, package_count, freight_amount_inr, attempt_count | No | Numeric measures are valid (positive, not null) |
| DQ-EVT-001 | Event/schema | All identity + timestamp + enum fields | No | Required fields present; enum values valid |

### Rules that need other tables (all available in workspace)

* **DQ-REF-001** joins to `silver_hubs`, `silver_carriers`, `silver_routes`
* **DQ-SCN-001** joins to `silver_scan_events` (shipment-level: flags a shipment if its linked scan events have invalid sequence numbers)
* **DQ-RTE-001** joins to `silver_routes` (only checks when route_id exists — if route is missing, that is DQ-REF-001)

In [ ]:
%sql
-- Step 1: Create candidate_shipments
-- This is a snapshot of silver_shipments.
-- The original silver_shipments table is NOT modified.

CREATE OR REPLACE TABLE candidate_shipments AS
SELECT * FROM silver_shipments;

-- Show the count
SELECT COUNT(*) AS candidate_row_count FROM candidate_shipments;

candidate_row_count
100020


In [ ]:
%sql
-- Step 2: Evaluate all 8 DQ rules per row
-- Creates an intermediate table dq_shipments_eval with:
--   * all original candidate columns
--   * a boolean flag for each DQ rule (fail_dq_xxx)
-- This table is then split into trusted and quarantine.

CREATE OR REPLACE TABLE dq_shipments_eval AS
WITH scan_issues AS (
  -- 1. event_sequence_no is NULL or < 1
  SELECT DISTINCT reference_id AS shipment_id
  FROM silver_scan_events
  WHERE event_sequence_no IS NULL OR event_sequence_no < 1

  UNION

  -- 2. Sequence does not start at 1 (MIN > 1 means first scan is not sequence 1)
  SELECT reference_id AS shipment_id
  FROM silver_scan_events
  WHERE event_sequence_no IS NOT NULL AND event_sequence_no >= 1
  GROUP BY reference_id
  HAVING MIN(event_sequence_no) > 1

  UNION

  -- 3. Gaps in sequence numbering (range does not match distinct count)
  SELECT reference_id AS shipment_id
  FROM silver_scan_events
  WHERE event_sequence_no IS NOT NULL AND event_sequence_no >= 1
  GROUP BY reference_id
  HAVING MAX(event_sequence_no) - MIN(event_sequence_no) + 1 <> COUNT(DISTINCT event_sequence_no)

  UNION

  -- 4. event_timestamp does not increase with event_sequence_no
  --    (a later sequence number has an earlier timestamp than a preceding one)
  SELECT DISTINCT reference_id AS shipment_id
  FROM (
    SELECT reference_id, event_sequence_no, event_timestamp,
           LAG(event_timestamp) OVER (PARTITION BY reference_id ORDER BY event_sequence_no) AS prev_ts
    FROM silver_scan_events
    WHERE event_sequence_no IS NOT NULL AND event_sequence_no >= 1
  )
  WHERE prev_ts IS NOT NULL AND event_timestamp < prev_ts

  -- NOT IMPLEMENTABLE: event_type progression ordering (CREATED -> PICKED_UP -> DEPARTED_HUB -> ...)
  -- The 8 event types (CREATED, PICKED_UP, DEPARTED_HUB, ARRIVED_HUB, STATUS_UPDATE,
  -- DELIVERY_CONFIRMATION, EXCEPTION, RETURN_CONFIRMATION) do not have a strict linear
  -- progression that can be objectively enforced from the schema. EXCEPTION and
  -- STATUS_UPDATE can occur at any point. Multiple DEPARTED_HUB/ARRIVED_HUB events occur
  -- as shipments transit between hubs. No documented business progression order exists.
),
dq_base AS (
  SELECT
    c.source_record_id, c.shipment_id, c.order_reference, c.route_id,
    c.origin_hub_id, c.destination_hub_id, c.carrier_id, c.service_level, c.transport_mode,
    c.booking_ts, c.pickup_ts, c.promised_delivery_ts, c.actual_delivery_ts, c.return_completed_ts,
    c.latest_status, c.delivery_outcome, c.status_reason,
    c.package_weight_kg, c.package_count, c.freight_amount_inr, c.attempt_count,
    c.exception_flag, c.source_system, c.run_id, c.ingestion_timestamp, c.source_file_name, c.batch_id,

    -- Window function: count how many rows share the same shipment_id
    COUNT(*) OVER (PARTITION BY c.shipment_id) AS shipment_id_dup_count,

    -- Reference table existence columns (NULL if the referenced record does not exist)
    h_org.hub_id       AS ref_origin_hub_id,
    h_dst.hub_id       AS ref_dest_hub_id,
    car.carrier_id     AS ref_carrier_id,
    rte.route_id       AS ref_route_id,
    rte.origin_hub_id       AS ref_route_origin_hub_id,
    rte.destination_hub_id  AS ref_route_dest_hub_id,
    rte.active_status        AS ref_route_active_status,
    rte.service_level        AS ref_route_service_level,
    rte.transport_mode       AS ref_route_transport_mode,
    rte.effective_start_date AS ref_route_effective_start_date,
    rte.effective_end_date   AS ref_route_effective_end_date,
    si.shipment_id     AS bad_scan_shipment_id
  FROM candidate_shipments c
  LEFT JOIN silver_hubs h_org    ON c.origin_hub_id      = h_org.hub_id
  LEFT JOIN silver_hubs h_dst    ON c.destination_hub_id = h_dst.hub_id
  LEFT JOIN silver_carriers car  ON c.carrier_id          = car.carrier_id
  LEFT JOIN silver_routes rte    ON c.route_id            = rte.route_id
  LEFT JOIN scan_issues si       ON c.shipment_id         = si.shipment_id
)
SELECT
  *,

  -- DQ-SHP-001: Shipment identity
  -- Fails if any identity field is NULL or shipment_id is duplicated
  (source_record_id IS NULL
    OR shipment_id IS NULL
    OR order_reference IS NULL
    OR shipment_id_dup_count > 1) AS fail_dq_shp_001,

  -- DQ-REF-001: Reference integrity
  -- Fails if any referenced ID does not exist in its reference table
  ((origin_hub_id IS NOT NULL AND ref_origin_hub_id IS NULL)
    OR (destination_hub_id IS NOT NULL AND ref_dest_hub_id IS NULL)
    OR (carrier_id IS NOT NULL AND ref_carrier_id IS NULL)
    OR (route_id IS NOT NULL AND ref_route_id IS NULL)) AS fail_dq_ref_001,

  -- DQ-TIM-001: Chronology (FULL)
  -- Fails if timestamps violate logical business order
  -- Checks: pickup >= booking, delivery >= pickup, return >= delivery,
  --          promised_delivery >= booking, promised_delivery >= pickup (when pickup exists)
  -- NULL timestamps are allowed for early-stage / non-delivered shipments
  ((pickup_ts IS NOT NULL AND pickup_ts < booking_ts)
    OR (actual_delivery_ts IS NOT NULL AND pickup_ts IS NOT NULL AND actual_delivery_ts < pickup_ts)
    OR (return_completed_ts IS NOT NULL AND actual_delivery_ts IS NOT NULL AND return_completed_ts < actual_delivery_ts)
    OR (promised_delivery_ts IS NOT NULL AND promised_delivery_ts < booking_ts)
    OR (promised_delivery_ts IS NOT NULL AND pickup_ts IS NOT NULL AND promised_delivery_ts < pickup_ts)) AS fail_dq_tim_001,

  -- DQ-SCN-001: Scan sequence (FULL)
  -- Fails if linked scan events have: NULL/negative sequence, no start at 1,
  -- gaps in sequence numbering, or timestamp order disagrees with sequence order
  -- NOT IMPLEMENTABLE: event_type progression ordering (see scan_issues CTE comment)
  (bad_scan_shipment_id IS NOT NULL) AS fail_dq_scn_001,

  -- DQ-RTE-001: Route consistency (FULL)
  -- Fails if shipment hubs, service_level, transport_mode do not match the route,
  -- or route is not active, or booking_ts falls outside the route's effective date range
  -- (only checked when the route exists; missing route is DQ-REF-001)
  -- NULL effective dates: check is skipped when date is NULL (all 107 routes have non-NULL dates)
  -- Boundary: booking ON effective_start_date or effective_end_date is allowed (inclusive)
  ((ref_route_id IS NOT NULL AND origin_hub_id <> ref_route_origin_hub_id)
    OR (ref_route_id IS NOT NULL AND destination_hub_id <> ref_route_dest_hub_id)
    OR (ref_route_id IS NOT NULL AND ref_route_active_status <> 'ACTIVE')
    OR (ref_route_id IS NOT NULL AND service_level <> ref_route_service_level)
    OR (ref_route_id IS NOT NULL AND transport_mode <> ref_route_transport_mode)
    OR (ref_route_id IS NOT NULL AND ref_route_effective_start_date IS NOT NULL
        AND CAST(booking_ts AS DATE) < ref_route_effective_start_date)
    OR (ref_route_id IS NOT NULL AND ref_route_effective_end_date IS NOT NULL
        AND CAST(booking_ts AS DATE) > ref_route_effective_end_date)) AS fail_dq_rte_001,

  -- DQ-DEL-001: Delivery/status consistency
  -- Fails if DELIVERED but no delivery timestamp, RETURNED but no return timestamp,
  -- or latest_status does not match delivery_outcome
  ((UPPER(latest_status) = 'DELIVERED' AND actual_delivery_ts IS NULL)
    OR (UPPER(latest_status) = 'RETURNED' AND return_completed_ts IS NULL)
    OR (UPPER(latest_status) <> UPPER(delivery_outcome))) AS fail_dq_del_001,

  -- DQ-MEA-001: Measures
  -- Fails if numeric measures are NULL or out of valid range
  (package_weight_kg IS NULL OR package_weight_kg <= 0
    OR package_count IS NULL OR package_count <= 0
    OR freight_amount_inr IS NULL OR freight_amount_inr < 0
    OR attempt_count < 0) AS fail_dq_mea_001,

  -- DQ-EVT-001: Event/schema
  -- Fails if required fields are missing or enum values are invalid
  (source_record_id IS NULL OR shipment_id IS NULL OR order_reference IS NULL
    OR booking_ts IS NULL OR promised_delivery_ts IS NULL
    OR latest_status NOT IN ('DELIVERED','IN_TRANSIT','AT_HUB','OUT_FOR_DELIVERY','RETURNED','CANCELLED')
    OR service_level NOT IN ('EXPRESS','STANDARD','PRIORITY')
    OR transport_mode NOT IN ('ROAD','AIR','RAIL')) AS fail_dq_evt_001
FROM dq_base;

-- Quick count: how many rows failed at least one rule
SELECT
  COUNT(*) AS total_evaluated,
  SUM(CASE WHEN fail_dq_shp_001 OR fail_dq_ref_001 OR fail_dq_tim_001 OR fail_dq_scn_001
            OR fail_dq_rte_001 OR fail_dq_del_001 OR fail_dq_mea_001 OR fail_dq_evt_001
       THEN 1 ELSE 0 END) AS total_failed,
  SUM(CASE WHEN NOT (fail_dq_shp_001 OR fail_dq_ref_001 OR fail_dq_tim_001 OR fail_dq_scn_001
            OR fail_dq_rte_001 OR fail_dq_del_001 OR fail_dq_mea_001 OR fail_dq_evt_001)
       THEN 1 ELSE 0 END) AS total_passed
FROM dq_shipments_eval;

total_evaluated,total_failed,total_passed
100020,163,99857


In [ ]:
%sql
-- Step 3: Create quarantine_shipments
-- Each row that failed at least one DQ rule goes here.
-- CRITICAL: One row per source_record_id — even if multiple rules failed.
-- Multiple failed rules are combined into a single failed_rule_ids ARRAY.

CREATE OR REPLACE TABLE quarantine_shipments AS
SELECT
  source_record_id,
  shipment_id,

  -- Multi-rule aggregation: collect ALL failed rule IDs into one array
  -- ARRAY_COMPACT removes NULL entries so only failed rules remain
  ARRAY_COMPACT(ARRAY(
    IF(fail_dq_shp_001, 'DQ-SHP-001', NULL),
    IF(fail_dq_ref_001, 'DQ-REF-001', NULL),
    IF(fail_dq_tim_001, 'DQ-TIM-001', NULL),
    IF(fail_dq_scn_001, 'DQ-SCN-001', NULL),
    IF(fail_dq_rte_001, 'DQ-RTE-001', NULL),
    IF(fail_dq_del_001, 'DQ-DEL-001', NULL),
    IF(fail_dq_mea_001, 'DQ-MEA-001', NULL),
    IF(fail_dq_evt_001, 'DQ-EVT-001', NULL)
  )) AS failed_rule_ids,

  -- Human-readable failure details — one line per failed rule
  CONCAT_WS(' | ',
    IF(fail_dq_shp_001, 'DQ-SHP-001: identity/uniqueness check failed', NULL),
    IF(fail_dq_ref_001, 'DQ-REF-001: reference integrity failed', NULL),
    IF(fail_dq_tim_001, 'DQ-TIM-001: timestamp chronology violated', NULL),
    IF(fail_dq_scn_001, 'DQ-SCN-001: linked scan has invalid sequence', NULL),
    IF(fail_dq_rte_001, 'DQ-RTE-001: route hub mismatch', NULL),
    IF(fail_dq_del_001, 'DQ-DEL-001: delivery/status inconsistency', NULL),
    IF(fail_dq_mea_001, 'DQ-MEA-001: invalid measure value', NULL),
    IF(fail_dq_evt_001, 'DQ-EVT-001: schema/enum validation failed', NULL)
  ) AS failure_details,

  -- Source/run information for traceability
  source_system,
  run_id,
  source_file_name,
  batch_id,
  ingestion_timestamp,

  -- All original shipment fields preserved for evidence
  order_reference,
  route_id,
  origin_hub_id,
  destination_hub_id,
  carrier_id,
  service_level,
  transport_mode,
  booking_ts,
  pickup_ts,
  promised_delivery_ts,
  actual_delivery_ts,
  return_completed_ts,
  latest_status,
  delivery_outcome,
  status_reason,
  package_weight_kg,
  package_count,
  freight_amount_inr,
  attempt_count,
  exception_flag
FROM dq_shipments_eval
WHERE fail_dq_shp_001 OR fail_dq_ref_001 OR fail_dq_tim_001 OR fail_dq_scn_001
   OR fail_dq_rte_001 OR fail_dq_del_001 OR fail_dq_mea_001 OR fail_dq_evt_001;

-- Show quarantine count
SELECT COUNT(*) AS quarantine_row_count FROM quarantine_shipments;

quarantine_row_count
163


In [ ]:
%sql
-- Step 4: Create trusted_shipments
-- A record enters Trusted ONLY when all 8 DQ rules pass.
-- The trusted table contains only the original shipment columns (no DQ helper columns).

CREATE OR REPLACE TABLE trusted_shipments AS
SELECT
  source_record_id, shipment_id, order_reference, route_id,
  origin_hub_id, destination_hub_id, carrier_id, service_level, transport_mode,
  booking_ts, pickup_ts, promised_delivery_ts, actual_delivery_ts, return_completed_ts,
  latest_status, delivery_outcome, status_reason,
  package_weight_kg, package_count, freight_amount_inr, attempt_count,
  exception_flag, source_system, run_id, ingestion_timestamp, source_file_name, batch_id
FROM dq_shipments_eval
WHERE NOT (fail_dq_shp_001 OR fail_dq_ref_001 OR fail_dq_tim_001 OR fail_dq_scn_001
        OR fail_dq_rte_001 OR fail_dq_del_001 OR fail_dq_mea_001 OR fail_dq_evt_001);

-- Show trusted count
SELECT COUNT(*) AS trusted_row_count FROM trusted_shipments;

trusted_row_count
99857


In [ ]:
%sql
-- Step 5: Reconciliation Check 1
-- Prove: Candidate distinct source_record_id = Trusted distinct source_record_id + Quarantine distinct source_record_id
-- The result must NOT be hard-coded — it is calculated from actual table counts.

SELECT
  (SELECT COUNT(DISTINCT source_record_id) FROM candidate_shipments)  AS candidate_distinct,
  (SELECT COUNT(DISTINCT source_record_id) FROM trusted_shipments)    AS trusted_distinct,
  (SELECT COUNT(DISTINCT source_record_id) FROM quarantine_shipments) AS quarantine_distinct,
  (SELECT COUNT(DISTINCT source_record_id) FROM candidate_shipments)
    - (SELECT COUNT(DISTINCT source_record_id) FROM trusted_shipments)
    - (SELECT COUNT(DISTINCT source_record_id) FROM quarantine_shipments) AS difference,
  CASE
    WHEN (SELECT COUNT(DISTINCT source_record_id) FROM candidate_shipments)
       = (SELECT COUNT(DISTINCT source_record_id) FROM trusted_shipments)
       + (SELECT COUNT(DISTINCT source_record_id) FROM quarantine_shipments)
    THEN 'PASS'
    ELSE 'FAIL'
  END AS reconciliation_result;

candidate_distinct,trusted_distinct,quarantine_distinct,difference,reconciliation_result
100020,99857,163,0,PASS


In [ ]:
%sql
-- Step 6: Reconciliation Check 2
-- Prove: Trusted intersection Quarantine = 0 (no source_record_id appears in both)
-- The result must NOT be hard-coded — it is calculated from an actual join.

SELECT
  COUNT(*) AS overlap_count,
  CASE
    WHEN COUNT(*) = 0 THEN 'PASS'
    ELSE 'FAIL'
  END AS overlap_result
FROM trusted_shipments t
INNER JOIN quarantine_shipments q ON t.source_record_id = q.source_record_id;

overlap_count,overlap_result
0,PASS


In [ ]:
%sql
-- Step 7: DQ Summary — failed count per rule
-- Uses the actual boolean flags from dq_shipments_eval.
-- Counts are NOT hard-coded — they are computed from the data.

SELECT 'DQ-SHP-001' AS rule_id, 'Shipment identity' AS rule_name,
       SUM(CASE WHEN fail_dq_shp_001 THEN 1 ELSE 0 END) AS failed_count
FROM dq_shipments_eval
UNION ALL
SELECT 'DQ-REF-001', 'Reference integrity',
       SUM(CASE WHEN fail_dq_ref_001 THEN 1 ELSE 0 END)
FROM dq_shipments_eval
UNION ALL
SELECT 'DQ-TIM-001', 'Chronology',
       SUM(CASE WHEN fail_dq_tim_001 THEN 1 ELSE 0 END)
FROM dq_shipments_eval
UNION ALL
SELECT 'DQ-SCN-001', 'Scan sequence',
       SUM(CASE WHEN fail_dq_scn_001 THEN 1 ELSE 0 END)
FROM dq_shipments_eval
UNION ALL
SELECT 'DQ-RTE-001', 'Route consistency',
       SUM(CASE WHEN fail_dq_rte_001 THEN 1 ELSE 0 END)
FROM dq_shipments_eval
UNION ALL
SELECT 'DQ-DEL-001', 'Delivery/status consistency',
       SUM(CASE WHEN fail_dq_del_001 THEN 1 ELSE 0 END)
FROM dq_shipments_eval
UNION ALL
SELECT 'DQ-MEA-001', 'Measures',
       SUM(CASE WHEN fail_dq_mea_001 THEN 1 ELSE 0 END)
FROM dq_shipments_eval
UNION ALL
SELECT 'DQ-EVT-001', 'Event/schema',
       SUM(CASE WHEN fail_dq_evt_001 THEN 1 ELSE 0 END)
FROM dq_shipments_eval
ORDER BY failed_count DESC;

rule_id,rule_name,failed_count
DQ-REF-001,Reference integrity,34
DQ-SHP-001,Shipment identity,32
DQ-TIM-001,Chronology,30
DQ-SCN-001,Scan sequence,28
DQ-RTE-001,Route consistency,22
DQ-MEA-001,Measures,20
DQ-DEL-001,Delivery/status consistency,10
DQ-EVT-001,Event/schema,0


In [ ]:
%sql
-- Step 8: Sample quarantine records — verify multi-rule aggregation
-- Displays actual quarantine records after execution.
-- Each row should have failed_rule_ids as an ARRAY.
-- Records with multiple failures should have ONE row with multiple rule IDs.
-- Records are ordered by number of failed rules (most failures first).

SELECT
  source_record_id,
  shipment_id,
  failed_rule_ids,
  failure_details
FROM quarantine_shipments
ORDER BY SIZE(failed_rule_ids) DESC, source_record_id
LIMIT 20;

source_record_id,shipment_id,failed_rule_ids,failure_details
SHPREC00004220,SHP00004220,"[""DQ-REF-001"", ""DQ-RTE-001""]",DQ-REF-001: reference integrity failed | DQ-RTE-001: route hub mismatch
SHPREC00017199,SHP00017199,"[""DQ-REF-001"", ""DQ-RTE-001""]",DQ-REF-001: reference integrity failed | DQ-RTE-001: route hub mismatch
SHPREC00029604,SHP00029604,"[""DQ-REF-001"", ""DQ-RTE-001""]",DQ-REF-001: reference integrity failed | DQ-RTE-001: route hub mismatch
SHPREC00036464,SHP00036464,"[""DQ-REF-001"", ""DQ-RTE-001""]",DQ-REF-001: reference integrity failed | DQ-RTE-001: route hub mismatch
SHPREC00039081,SHP00039081,"[""DQ-REF-001"", ""DQ-RTE-001""]",DQ-REF-001: reference integrity failed | DQ-RTE-001: route hub mismatch
SHPREC00042860,SHP00042860,"[""DQ-REF-001"", ""DQ-RTE-001""]",DQ-REF-001: reference integrity failed | DQ-RTE-001: route hub mismatch
SHPREC00062186,SHP00062186,"[""DQ-REF-001"", ""DQ-RTE-001""]",DQ-REF-001: reference integrity failed | DQ-RTE-001: route hub mismatch
SHPREC00063151,SHP00063151,"[""DQ-REF-001"", ""DQ-RTE-001""]",DQ-REF-001: reference integrity failed | DQ-RTE-001: route hub mismatch
SHPREC00079951,SHP00079951,"[""DQ-REF-001"", ""DQ-RTE-001""]",DQ-REF-001: reference integrity failed | DQ-RTE-001: route hub mismatch
SHPREC00082237,SHP00082237,"[""DQ-REF-001"", ""DQ-RTE-001""]",DQ-REF-001: reference integrity failed | DQ-RTE-001: route hub mismatch


## Verification Checklist

Run the cells in order (top to bottom) and check these outputs:

| Cell | What to check | Expected result |
|------|--------------|-----------------|
| Step 1 (candidate) | `candidate_row_count` | Should equal the row count of silver_shipments |
| Step 2 (dq_eval) | `total_evaluated` | Should equal candidate_row_count |
| | `total_failed` | Should be > 0 (we know there are bad records) |
| | `total_passed` | Should be total_evaluated minus total_failed |
| Step 3 (quarantine) | `quarantine_row_count` | Should equal total_failed from Step 2 |
| Step 4 (trusted) | `trusted_row_count` | Should equal total_passed from Step 2 |
| Step 5 (recon 1) | `reconciliation_result` | Should say **PASS** |
| | `difference` | Should be **0** |
| Step 6 (recon 2) | `overlap_result` | Should say **PASS** |
| | `overlap_count` | Should be **0** |
| Step 7 (summary) | Failed counts per rule | Should show actual failure counts per rule ID |
| Step 8 (samples) | `failed_rule_ids` | Should be an ARRAY — some rows may have multiple rule IDs |

### Multi-rule aggregation

The `failed_rule_ids` column in quarantine_shipments is an ARRAY. A single record that violates
both DQ-MEA-001 and DQ-TIM-001 will have: `["DQ-MEA-001", "DQ-TIM-001"]` — ONE row, not two.
This is achieved using `ARRAY_COMPACT(ARRAY(IF(flag, 'rule_id', NULL), ...))` which collects
all failed rules into one array per source_record_id.

### Rule implementation status

| Rule ID | Status | Notes |
|---------|--------|-------|
| DQ-SHP-001 | FULL | Identity fields not NULL + shipment_id uniqueness |
| DQ-REF-001 | FULL | All 4 reference types checked against actual reference tables |
| DQ-TIM-001 | FULL | All 5 timestamp chronology checks including promised_delivery_ts |
| DQ-SCN-001 | FULL | NULL/neg sequence, start-at-1, gaps, timestamp ordering. Event_type progression NOT IMPLEMENTABLE (no documented business order) |
| DQ-RTE-001 | FULL | Hub match + active_status + service_level + transport_mode + effective date range |
| DQ-DEL-001 | FULL | All 3 delivery/status consistency conditions |
| DQ-MEA-001 | FULL | All 4 measure fields validated |
| DQ-EVT-001 | FULL | Required fields + enum validation |

All 8 rules are now FULL. DQ-SCN-001 event_type progression remains NOT IMPLEMENTABLE
(no documented business progression order exists in the schema).

### Rerun safety

All table-creation cells use `CREATE OR REPLACE TABLE`, so the notebook can be safely
run again without accumulating duplicate rows.

### Extending to other entities

To add the same framework for hubs, carriers, routes, scans, and exceptions,
copy this notebook pattern and adjust the source table, DQ rules, and column names.

In [ ]:
%sql
-- Step 8: Sample quarantine records — verify multi-rule aggregation
-- Displays actual quarantine records after execution.
-- Each row should have failed_rule_ids as an ARRAY.
-- Records with multiple failures should have ONE row with multiple rule IDs.
-- Records are ordered by number of failed rules (most failures first).

SELECT
  source_record_id,
  shipment_id,
  failed_rule_ids,
  failure_details
FROM quarantine_shipments
ORDER BY SIZE(failed_rule_ids) DESC, source_record_id
LIMIT 20;

source_record_id,shipment_id,failed_rule_ids,failure_details
SHPREC00004220,SHP00004220,"[""DQ-REF-001"", ""DQ-RTE-001""]",DQ-REF-001: reference integrity failed | DQ-RTE-001: route hub mismatch
SHPREC00017199,SHP00017199,"[""DQ-REF-001"", ""DQ-RTE-001""]",DQ-REF-001: reference integrity failed | DQ-RTE-001: route hub mismatch
SHPREC00029604,SHP00029604,"[""DQ-REF-001"", ""DQ-RTE-001""]",DQ-REF-001: reference integrity failed | DQ-RTE-001: route hub mismatch
SHPREC00036464,SHP00036464,"[""DQ-REF-001"", ""DQ-RTE-001""]",DQ-REF-001: reference integrity failed | DQ-RTE-001: route hub mismatch
SHPREC00039081,SHP00039081,"[""DQ-REF-001"", ""DQ-RTE-001""]",DQ-REF-001: reference integrity failed | DQ-RTE-001: route hub mismatch
SHPREC00042860,SHP00042860,"[""DQ-REF-001"", ""DQ-RTE-001""]",DQ-REF-001: reference integrity failed | DQ-RTE-001: route hub mismatch
SHPREC00062186,SHP00062186,"[""DQ-REF-001"", ""DQ-RTE-001""]",DQ-REF-001: reference integrity failed | DQ-RTE-001: route hub mismatch
SHPREC00063151,SHP00063151,"[""DQ-REF-001"", ""DQ-RTE-001""]",DQ-REF-001: reference integrity failed | DQ-RTE-001: route hub mismatch
SHPREC00079951,SHP00079951,"[""DQ-REF-001"", ""DQ-RTE-001""]",DQ-REF-001: reference integrity failed | DQ-RTE-001: route hub mismatch
SHPREC00082237,SHP00082237,"[""DQ-REF-001"", ""DQ-RTE-001""]",DQ-REF-001: reference integrity failed | DQ-RTE-001: route hub mismatch


## Verification Checklist

Run the cells in order (top to bottom) and check these outputs:

| Cell | What to check | Expected result |
|------|--------------|-----------------|
| Step 1 (candidate) | `candidate_row_count` | Should equal the row count of silver_shipments |
| Step 2 (dq_eval) | `total_evaluated` | Should equal candidate_row_count |
| | `total_failed` | Should be > 0 (we know there are bad records) |
| | `total_passed` | Should be total_evaluated minus total_failed |
| Step 3 (quarantine) | `quarantine_row_count` | Should equal total_failed from Step 2 |
| Step 4 (trusted) | `trusted_row_count` | Should equal total_passed from Step 2 |
| Step 5 (recon 1) | `reconciliation_result` | Should say **PASS** |
| | `difference` | Should be **0** |
| Step 6 (recon 2) | `overlap_result` | Should say **PASS** |
| | `overlap_count` | Should be **0** |
| Step 7 (summary) | Failed counts per rule | Should show actual failure counts per rule ID |
| Step 8 (samples) | `failed_rule_ids` | Should be an ARRAY — some rows may have multiple rule IDs |

### Multi-rule aggregation

The `failed_rule_ids` column in quarantine_shipments is an ARRAY. A single record that violates
both DQ-MEA-001 and DQ-TIM-001 will have: `["DQ-MEA-001", "DQ-TIM-001"]` — ONE row, not two.
This is achieved using `ARRAY_COMPACT(ARRAY(IF(flag, 'rule_id', NULL), ...))` which collects
all failed rules into one array per source_record_id.

### Rule implementation status

| Rule ID | Status | Notes |
|---------|--------|-------|
| DQ-SHP-001 | FULL | Identity fields not NULL + shipment_id uniqueness |
| DQ-REF-001 | FULL | All 4 reference types checked against actual reference tables |
| DQ-TIM-001 | PARTIAL | Does not check promised_delivery_ts chronology |
| DQ-SCN-001 | PARTIAL | Only checks event_sequence_no >= 1, not full scan ordering |
| DQ-RTE-001 | PARTIAL | Only checks hub matching, not service_level/active_status |
| DQ-DEL-001 | FULL | All 3 delivery/status consistency conditions |
| DQ-MEA-001 | FULL | All 4 measure fields validated |
| DQ-EVT-001 | FULL | Required fields + enum validation |

The PARTIAL rules will be improved separately after the Candidate to Trusted/Quarantine framework is verified.

### Rerun safety

All table-creation cells use `CREATE OR REPLACE TABLE`, so the notebook can be safely
run again without accumulating duplicate rows.

### Extending to other entities

To add the same framework for hubs, carriers, routes, scans, and exceptions,
copy this notebook pattern and adjust the source table, DQ rules, and column names.

In [ ]:
%sql
SELECT
    'Candidate' AS status,
    COUNT(*) AS row_count
FROM silver_shipments

UNION ALL

SELECT
    'Trusted',
    COUNT(*)
FROM trusted_shipments

UNION ALL

SELECT
    'Quarantine',
    COUNT(*)
FROM quarantine_shipments;

status,row_count
Candidate,100020
Trusted,99857
Quarantine,163


In [ ]:
%sql
SELECT
    COUNT(*) AS total_evaluated,
    SUM(CASE WHEN fail_dq_shp_001 OR fail_dq_ref_001 OR fail_dq_tim_001
                  OR fail_dq_scn_001 OR fail_dq_rte_001 OR fail_dq_del_001
                  OR fail_dq_mea_001 OR fail_dq_evt_001
             THEN 1 ELSE 0 END) AS total_failed,
    SUM(CASE WHEN NOT (fail_dq_shp_001 OR fail_dq_ref_001 OR fail_dq_tim_001
                  OR fail_dq_scn_001 OR fail_dq_rte_001 OR fail_dq_del_001
                  OR fail_dq_mea_001 OR fail_dq_evt_001)
             THEN 1 ELSE 0 END) AS total_passed
FROM dq_shipments_eval;

total_evaluated,total_failed,total_passed
100020,163,99857


In [ ]:
%sql
SELECT
    (SELECT COUNT(DISTINCT source_record_id) FROM candidate_shipments) AS candidate_distinct,
    (SELECT COUNT(DISTINCT source_record_id) FROM trusted_shipments) AS trusted_distinct,
    (SELECT COUNT(DISTINCT source_record_id) FROM quarantine_shipments) AS quarantine_distinct,
    (SELECT COUNT(DISTINCT source_record_id) FROM candidate_shipments)
      - (SELECT COUNT(DISTINCT source_record_id) FROM trusted_shipments)
      - (SELECT COUNT(DISTINCT source_record_id) FROM quarantine_shipments) AS difference,
    CASE
        WHEN (SELECT COUNT(DISTINCT source_record_id) FROM candidate_shipments)
           = (SELECT COUNT(DISTINCT source_record_id) FROM trusted_shipments)
           + (SELECT COUNT(DISTINCT source_record_id) FROM quarantine_shipments)
        THEN 'PASS'
        ELSE 'FAIL'
    END AS reconciliation_result;

candidate_distinct,trusted_distinct,quarantine_distinct,difference,reconciliation_result
100020,99857,163,0,PASS


In [ ]:
%sql
SELECT
    COUNT(*) AS overlap_count,
    CASE
        WHEN COUNT(*) = 0 THEN 'PASS'
        ELSE 'FAIL'
    END AS overlap_result
FROM trusted_shipments t
INNER JOIN quarantine_shipments q
    ON t.source_record_id = q.source_record_id;

overlap_count,overlap_result
0,PASS


In [ ]:
%sql
SELECT
    'DQ-SHP-001' AS rule_id,
    SUM(CASE WHEN fail_dq_shp_001 THEN 1 ELSE 0 END) AS failed_count
FROM dq_shipments_eval

UNION ALL
SELECT 'DQ-REF-001', SUM(CASE WHEN fail_dq_ref_001 THEN 1 ELSE 0 END) FROM dq_shipments_eval
UNION ALL
SELECT 'DQ-TIM-001', SUM(CASE WHEN fail_dq_tim_001 THEN 1 ELSE 0 END) FROM dq_shipments_eval
UNION ALL
SELECT 'DQ-SCN-001', SUM(CASE WHEN fail_dq_scn_001 THEN 1 ELSE 0 END) FROM dq_shipments_eval
UNION ALL
SELECT 'DQ-RTE-001', SUM(CASE WHEN fail_dq_rte_001 THEN 1 ELSE 0 END) FROM dq_shipments_eval
UNION ALL
SELECT 'DQ-DEL-001', SUM(CASE WHEN fail_dq_del_001 THEN 1 ELSE 0 END) FROM dq_shipments_eval
UNION ALL
SELECT 'DQ-MEA-001', SUM(CASE WHEN fail_dq_mea_001 THEN 1 ELSE 0 END) FROM dq_shipments_eval
UNION ALL
SELECT 'DQ-EVT-001', SUM(CASE WHEN fail_dq_evt_001 THEN 1 ELSE 0 END) FROM dq_shipments_eval;

rule_id,failed_count
DQ-SHP-001,32
DQ-REF-001,34
DQ-TIM-001,30
DQ-SCN-001,28
DQ-RTE-001,22
DQ-DEL-001,10
DQ-MEA-001,20
DQ-EVT-001,0


In [ ]:
%sql
SELECT
    source_record_id,
    shipment_id,
    failed_rule_ids,
    failure_details
FROM quarantine_shipments
ORDER BY SIZE(failed_rule_ids) DESC
LIMIT 20;

source_record_id,shipment_id,failed_rule_ids,failure_details
SHPREC00093818,SHP00093818,"[""DQ-REF-001"", ""DQ-RTE-001""]",DQ-REF-001: reference integrity failed | DQ-RTE-001: route hub mismatch
SHPREC00029604,SHP00029604,"[""DQ-REF-001"", ""DQ-RTE-001""]",DQ-REF-001: reference integrity failed | DQ-RTE-001: route hub mismatch
SHPREC00063151,SHP00063151,"[""DQ-REF-001"", ""DQ-RTE-001""]",DQ-REF-001: reference integrity failed | DQ-RTE-001: route hub mismatch
SHPREC00086474,SHP00086474,"[""DQ-SCN-001"", ""DQ-MEA-001""]",DQ-SCN-001: linked scan has invalid sequence | DQ-MEA-001: invalid measure value
SHPREC00042860,SHP00042860,"[""DQ-REF-001"", ""DQ-RTE-001""]",DQ-REF-001: reference integrity failed | DQ-RTE-001: route hub mismatch
SHPREC00017199,SHP00017199,"[""DQ-REF-001"", ""DQ-RTE-001""]",DQ-REF-001: reference integrity failed | DQ-RTE-001: route hub mismatch
SHPREC00036464,SHP00036464,"[""DQ-REF-001"", ""DQ-RTE-001""]",DQ-REF-001: reference integrity failed | DQ-RTE-001: route hub mismatch
SHPREC00062186,SHP00062186,"[""DQ-REF-001"", ""DQ-RTE-001""]",DQ-REF-001: reference integrity failed | DQ-RTE-001: route hub mismatch
SHPREC00092755,SHP00092755,"[""DQ-REF-001"", ""DQ-RTE-001""]",DQ-REF-001: reference integrity failed | DQ-RTE-001: route hub mismatch
SHPREC00079951,SHP00079951,"[""DQ-REF-001"", ""DQ-RTE-001""]",DQ-REF-001: reference integrity failed | DQ-RTE-001: route hub mismatch


## Step 9 — Controlled Correction + Replay

Demonstrates the correction workflow: Quarantine -> Correction -> DQ Replay -> Trusted.

**Selected record:** SHPREC00001565 (fails only DQ-MEA-001: package_weight_kg = -4)
**Correction:** package_weight_kg changed from -4 to 4 (ABS — sign entry error)
**Method:** Uses TEMP VIEW only — does NOT modify Silver or production Trusted/Quarantine tables.

In [ ]:
%sql
-- Step 9a: BEFORE — Show the original quarantined record
SELECT 
  q.source_record_id,
  q.shipment_id,
  q.failed_rule_ids,
  q.failure_details,
  s.package_weight_kg AS original_weight_kg,
  s.package_count,
  s.freight_amount_inr,
  s.attempt_count
FROM quarantine_shipments q
LEFT JOIN silver_shipments s ON q.source_record_id = s.source_record_id
WHERE q.source_record_id = 'SHPREC00001565';

In [ ]:
%sql
-- Step 9b: CORRECTION — Create a corrected staging view (TEMP VIEW, not a table)
-- Does NOT modify silver_shipments or any production DQ table.
CREATE OR REPLACE TEMP VIEW corrected_shipment_demo AS
SELECT 
  source_record_id, shipment_id, order_reference, route_id,
  origin_hub_id, destination_hub_id, carrier_id, service_level, transport_mode,
  booking_ts, pickup_ts, promised_delivery_ts, actual_delivery_ts, return_completed_ts,
  latest_status, delivery_outcome, status_reason,
  ABS(package_weight_kg) AS package_weight_kg,  -- CORRECTION: -4 -> 4
  package_count, freight_amount_inr, attempt_count,
  exception_flag, source_system, run_id, ingestion_timestamp, source_file_name, batch_id
FROM silver_shipments
WHERE source_record_id = 'SHPREC00001565';

-- Step 9c: REPLAY — Rerun DQ-MEA-001 against the corrected record
SELECT
  source_record_id,
  shipment_id,
  package_weight_kg AS corrected_weight_kg,
  package_count,
  freight_amount_inr,
  attempt_count,
  (package_weight_kg IS NULL OR package_weight_kg <= 0
   OR package_count IS NULL OR package_count <= 0
   OR freight_amount_inr IS NULL OR freight_amount_inr < 0
   OR attempt_count < 0) AS fail_dq_mea_001_after_correction,
  CASE 
    WHEN NOT (package_weight_kg IS NULL OR package_weight_kg <= 0
         OR package_count IS NULL OR package_count <= 0
         OR freight_amount_inr IS NULL OR freight_amount_inr < 0
         OR attempt_count < 0)
    THEN 'PASSES — record would move from Quarantine to Trusted'
    ELSE 'STILL FAILS — remains in Quarantine'
  END AS replay_result
FROM corrected_shipment_demo;

---
# Part B — Hubs, Carriers, Routes, Scan events, Exceptions

# DQ Framework — Other Entities (Week 06)

Candidate → Trusted/Quarantine DQ framework for hubs, carriers, routes, scan events, and exceptions.
Each entity follows the same pattern as the shipments framework:
candidate → DQ evaluation → trusted / quarantine → reconciliation → overlap check.

Silver source tables are NOT modified. All tables use CREATE OR REPLACE for rerunnability.

In [ ]:
%sql
-- ============================================================================
-- ENTITY 1: HUBS (silver_hubs — 14 rows)
-- Rules: DQ-HUB-001 (identity), DQ-HUB-002 (schema/enum), DQ-HUB-003 (dates)
-- Shipment-specific rules DQ-SHP/REF/TIM/SCN/RTE/DEL/MEA/EVT do NOT apply to hubs.
-- ============================================================================

CREATE OR REPLACE TABLE candidate_hubs AS SELECT * FROM silver_hubs;

CREATE OR REPLACE TABLE dq_hubs_eval AS
SELECT
  *,
  COUNT(*) OVER (PARTITION BY hub_id) AS hub_id_dup_count,
  (source_record_id IS NULL OR hub_id IS NULL OR hub_name IS NULL
   OR COUNT(*) OVER (PARTITION BY hub_id) > 1) AS fail_dq_hub_001,
  (hub_type NOT IN ('GATEWAY','REGIONAL','TRANSIT','LAST_MILE')
   OR region NOT IN ('North','South','East','West','Central')
   OR active_status NOT IN ('ACTIVE','INACTIVE')
   OR capacity_band NOT IN ('HIGH','MEDIUM','LOW')) AS fail_dq_hub_002,
  (effective_start_date IS NOT NULL AND effective_end_date IS NOT NULL
   AND TRY_CAST(effective_start_date AS DATE) IS NOT NULL
   AND TRY_CAST(effective_end_date AS DATE) IS NOT NULL
   AND CAST(effective_start_date AS DATE) > CAST(effective_end_date AS DATE)) AS fail_dq_hub_003
FROM candidate_hubs;

CREATE OR REPLACE TABLE trusted_hubs AS
SELECT source_record_id, hub_id, hub_name, hub_type, region, city,
  active_status, capacity_band, effective_start_date, effective_end_date,
  ingestion_timestamp, source_system, source_file_name, batch_id
FROM dq_hubs_eval
WHERE NOT (fail_dq_hub_001 OR fail_dq_hub_002 OR fail_dq_hub_003);

CREATE OR REPLACE TABLE quarantine_hubs AS
SELECT source_record_id, hub_id, hub_name,
  ARRAY_COMPACT(ARRAY(
    IF(fail_dq_hub_001, 'DQ-HUB-001', NULL),
    IF(fail_dq_hub_002, 'DQ-HUB-002', NULL),
    IF(fail_dq_hub_003, 'DQ-HUB-003', NULL)
  )) AS failed_rule_ids,
  CONCAT_WS(' | ',
    IF(fail_dq_hub_001, 'DQ-HUB-001: identity/null/duplicate hub_id', NULL),
    IF(fail_dq_hub_002, 'DQ-HUB-002: invalid enum value', NULL),
    IF(fail_dq_hub_003, 'DQ-HUB-003: date chronology violation', NULL)
  ) AS failure_details
FROM dq_hubs_eval
WHERE fail_dq_hub_001 OR fail_dq_hub_002 OR fail_dq_hub_003;

SELECT
  (SELECT COUNT(*) FROM candidate_hubs) AS candidate_rows,
  (SELECT COUNT(*) FROM trusted_hubs) AS trusted_rows,
  (SELECT COUNT(*) FROM quarantine_hubs) AS quarantine_rows,
  (SELECT COUNT(*) FROM candidate_hubs) - (SELECT COUNT(*) FROM trusted_hubs) - (SELECT COUNT(*) FROM quarantine_hubs) AS difference,
  CASE WHEN (SELECT COUNT(*) FROM candidate_hubs) = (SELECT COUNT(*) FROM trusted_hubs) + (SELECT COUNT(*) FROM quarantine_hubs)
    THEN 'PASS' ELSE 'FAIL' END AS reconciliation_result,
  (SELECT COUNT(*) FROM trusted_hubs t INNER JOIN quarantine_hubs q ON t.source_record_id = q.source_record_id) AS overlap_count;

candidate_rows,trusted_rows,quarantine_rows,difference,reconciliation_result,overlap_count
14,14,0,0,PASS,0


In [ ]:
%sql
-- ============================================================================
-- ENTITY 2: CARRIERS (silver_carriers — 10 rows)
-- Rules: DQ-CAR-001 (identity), DQ-CAR-002 (schema/enum), DQ-CAR-003 (dates)
-- ============================================================================

CREATE OR REPLACE TABLE candidate_carriers AS SELECT * FROM silver_carriers;

CREATE OR REPLACE TABLE dq_carriers_eval AS
SELECT
  *,
  COUNT(*) OVER (PARTITION BY carrier_id) AS carrier_id_dup_count,
  (source_record_id IS NULL OR carrier_id IS NULL OR carrier_name IS NULL
   OR COUNT(*) OVER (PARTITION BY carrier_id) > 1) AS fail_dq_car_001,
  (transport_mode NOT IN ('ROAD','AIR','RAIL')
   OR primary_service_level NOT IN ('STANDARD','EXPRESS','PRIORITY')
   OR active_status NOT IN ('ACTIVE','INACTIVE')) AS fail_dq_car_002,
  (effective_start_date IS NOT NULL AND effective_end_date IS NOT NULL
   AND TRY_CAST(effective_start_date AS DATE) IS NOT NULL
   AND TRY_CAST(effective_end_date AS DATE) IS NOT NULL
   AND CAST(effective_start_date AS DATE) > CAST(effective_end_date AS DATE)) AS fail_dq_car_003
FROM candidate_carriers;

CREATE OR REPLACE TABLE trusted_carriers AS
SELECT source_record_id, carrier_id, carrier_name, transport_mode,
  primary_service_level, active_status, effective_start_date, effective_end_date,
  ingestion_timestamp, source_system, source_file_name, batch_id
FROM dq_carriers_eval
WHERE NOT (fail_dq_car_001 OR fail_dq_car_002 OR fail_dq_car_003);

CREATE OR REPLACE TABLE quarantine_carriers AS
SELECT source_record_id, carrier_id, carrier_name,
  ARRAY_COMPACT(ARRAY(
    IF(fail_dq_car_001, 'DQ-CAR-001', NULL),
    IF(fail_dq_car_002, 'DQ-CAR-002', NULL),
    IF(fail_dq_car_003, 'DQ-CAR-003', NULL)
  )) AS failed_rule_ids,
  CONCAT_WS(' | ',
    IF(fail_dq_car_001, 'DQ-CAR-001: identity/null/duplicate carrier_id', NULL),
    IF(fail_dq_car_002, 'DQ-CAR-002: invalid enum value', NULL),
    IF(fail_dq_car_003, 'DQ-CAR-003: date chronology violation', NULL)
  ) AS failure_details
FROM dq_carriers_eval
WHERE fail_dq_car_001 OR fail_dq_car_002 OR fail_dq_car_003;

SELECT
  (SELECT COUNT(*) FROM candidate_carriers) AS candidate_rows,
  (SELECT COUNT(*) FROM trusted_carriers) AS trusted_rows,
  (SELECT COUNT(*) FROM quarantine_carriers) AS quarantine_rows,
  (SELECT COUNT(*) FROM candidate_carriers) - (SELECT COUNT(*) FROM trusted_carriers) - (SELECT COUNT(*) FROM quarantine_carriers) AS difference,
  CASE WHEN (SELECT COUNT(*) FROM candidate_carriers) = (SELECT COUNT(*) FROM trusted_carriers) + (SELECT COUNT(*) FROM quarantine_carriers)
    THEN 'PASS' ELSE 'FAIL' END AS reconciliation_result,
  (SELECT COUNT(*) FROM trusted_carriers t INNER JOIN quarantine_carriers q ON t.source_record_id = q.source_record_id) AS overlap_count;

candidate_rows,trusted_rows,quarantine_rows,difference,reconciliation_result,overlap_count
10,10,0,0,PASS,0


In [ ]:
%sql
-- ============================================================================
-- ENTITY 3: ROUTES (silver_routes — 107 rows)
-- Rules: DQ-RT-001 (identity), DQ-RT-002 (reference), DQ-RT-003 (schema),
--        DQ-RT-004 (dates), DQ-RT-005 (measures)
-- ============================================================================

CREATE OR REPLACE TABLE candidate_routes AS SELECT * FROM silver_routes;

CREATE OR REPLACE TABLE dq_routes_eval AS
SELECT
  c.*,
  COUNT(*) OVER (PARTITION BY c.route_id) AS route_id_dup_count,
  h_org.hub_id AS ref_origin_hub_id,
  h_dst.hub_id AS ref_dest_hub_id,
  (c.source_record_id IS NULL OR c.route_id IS NULL
   OR COUNT(*) OVER (PARTITION BY c.route_id) > 1) AS fail_dq_rt_001,
  ((c.origin_hub_id IS NOT NULL AND h_org.hub_id IS NULL)
   OR (c.destination_hub_id IS NOT NULL AND h_dst.hub_id IS NULL)) AS fail_dq_rt_002,
  (c.service_level NOT IN ('STANDARD','EXPRESS','PRIORITY','OVERNIGHT')
   OR c.transport_mode NOT IN ('ROAD','AIR','RAIL')
   OR c.active_status NOT IN ('ACTIVE','INACTIVE')
   OR c.route_band NOT IN ('SHORT','MEDIUM','LONG')) AS fail_dq_rt_003,
  (c.effective_start_date IS NOT NULL AND c.effective_end_date IS NOT NULL
   AND TRY_CAST(c.effective_start_date AS DATE) IS NOT NULL
   AND TRY_CAST(c.effective_end_date AS DATE) IS NOT NULL
   AND CAST(c.effective_start_date AS DATE) > CAST(c.effective_end_date AS DATE)) AS fail_dq_rt_004,
  (c.distance_km IS NULL OR c.distance_km <= 0
   OR c.expected_transit_hours IS NULL OR c.expected_transit_hours <= 0) AS fail_dq_rt_005
FROM candidate_routes c
LEFT JOIN silver_hubs h_org ON c.origin_hub_id = h_org.hub_id
LEFT JOIN silver_hubs h_dst ON c.destination_hub_id = h_dst.hub_id;

CREATE OR REPLACE TABLE trusted_routes AS
SELECT source_record_id, route_id, origin_hub_id, destination_hub_id,
  service_level, transport_mode, distance_km, expected_transit_hours,
  route_band, active_status, effective_start_date, effective_end_date,
  ingestion_timestamp, source_system, source_file_name, batch_id
FROM dq_routes_eval
WHERE NOT (fail_dq_rt_001 OR fail_dq_rt_002 OR fail_dq_rt_003 OR fail_dq_rt_004 OR fail_dq_rt_005);

CREATE OR REPLACE TABLE quarantine_routes AS
SELECT source_record_id, route_id,
  ARRAY_COMPACT(ARRAY(
    IF(fail_dq_rt_001, 'DQ-RT-001', NULL),
    IF(fail_dq_rt_002, 'DQ-RT-002', NULL),
    IF(fail_dq_rt_003, 'DQ-RT-003', NULL),
    IF(fail_dq_rt_004, 'DQ-RT-004', NULL),
    IF(fail_dq_rt_005, 'DQ-RT-005', NULL)
  )) AS failed_rule_ids,
  CONCAT_WS(' | ',
    IF(fail_dq_rt_001, 'DQ-RT-001: identity/null/duplicate route_id', NULL),
    IF(fail_dq_rt_002, 'DQ-RT-002: referenced hub does not exist', NULL),
    IF(fail_dq_rt_003, 'DQ-RT-003: invalid enum value', NULL),
    IF(fail_dq_rt_004, 'DQ-RT-004: date chronology violation', NULL),
    IF(fail_dq_rt_005, 'DQ-RT-005: invalid measure value', NULL)
  ) AS failure_details
FROM dq_routes_eval
WHERE fail_dq_rt_001 OR fail_dq_rt_002 OR fail_dq_rt_003 OR fail_dq_rt_004 OR fail_dq_rt_005;

SELECT
  (SELECT COUNT(*) FROM candidate_routes) AS candidate_rows,
  (SELECT COUNT(*) FROM trusted_routes) AS trusted_rows,
  (SELECT COUNT(*) FROM quarantine_routes) AS quarantine_rows,
  (SELECT COUNT(*) FROM candidate_routes) - (SELECT COUNT(*) FROM trusted_routes) - (SELECT COUNT(*) FROM quarantine_routes) AS difference,
  CASE WHEN (SELECT COUNT(*) FROM candidate_routes) = (SELECT COUNT(*) FROM trusted_routes) + (SELECT COUNT(*) FROM quarantine_routes)
    THEN 'PASS' ELSE 'FAIL' END AS reconciliation_result,
  (SELECT COUNT(*) FROM trusted_routes t INNER JOIN quarantine_routes q ON t.source_record_id = q.source_record_id) AS overlap_count;

candidate_rows,trusted_rows,quarantine_rows,difference,reconciliation_result,overlap_count
107,103,4,0,PASS,0


In [ ]:
%sql
-- ============================================================================
-- ENTITY 4: SCAN EVENTS (silver_scan_events — 710666 rows)
-- Rules: DQ-SCN-001 (identity), DQ-SCN-002 (schema/enum), DQ-SCN-003 (timestamp)
-- Note: record_id is the physical source record key (not reference_id which is shipment_id FK)
-- ============================================================================

CREATE OR REPLACE TABLE candidate_scan_events AS SELECT * FROM silver_scan_events;

CREATE OR REPLACE TABLE dq_scan_events_eval AS
SELECT
  *,
  COUNT(*) OVER (PARTITION BY record_id) AS record_id_dup_count,
  (record_id IS NULL OR scan_id IS NULL
   OR COUNT(*) OVER (PARTITION BY record_id) > 1) AS fail_dq_scn_001,
  (event_type NOT IN ('CREATED','PICKED_UP','DEPARTED_HUB','ARRIVED_HUB','STATUS_UPDATE','DELIVERY_CONFIRMATION','EXCEPTION','RETURN_CONFIRMATION')
   OR status_standard IS NULL) AS fail_dq_scn_002,
  (event_timestamp IS NULL) AS fail_dq_scn_003
FROM candidate_scan_events;

CREATE OR REPLACE TABLE trusted_scan_events AS
SELECT record_id, reference_id, event_timestamp, event_date, event_type,
  status_standard, source_system, ingestion_timestamp, scan_id, event_sequence_no,
  hub_id, carrier_id, route_id, exception_type, package_condition, status_reason,
  run_id, source_file_name, batch_id
FROM dq_scan_events_eval
WHERE NOT (fail_dq_scn_001 OR fail_dq_scn_002 OR fail_dq_scn_003);

CREATE OR REPLACE TABLE quarantine_scan_events AS
SELECT record_id, reference_id, scan_id, event_type,
  ARRAY_COMPACT(ARRAY(
    IF(fail_dq_scn_001, 'DQ-SCN-001', NULL),
    IF(fail_dq_scn_002, 'DQ-SCN-002', NULL),
    IF(fail_dq_scn_003, 'DQ-SCN-003', NULL)
  )) AS failed_rule_ids,
  CONCAT_WS(' | ',
    IF(fail_dq_scn_001, 'DQ-SCN-001: identity/null/duplicate record_id', NULL),
    IF(fail_dq_scn_002, 'DQ-SCN-002: invalid event_type or null status', NULL),
    IF(fail_dq_scn_003, 'DQ-SCN-003: null event_timestamp', NULL)
  ) AS failure_details
FROM dq_scan_events_eval
WHERE fail_dq_scn_001 OR fail_dq_scn_002 OR fail_dq_scn_003;

SELECT
  (SELECT COUNT(*) FROM candidate_scan_events) AS candidate_rows,
  (SELECT COUNT(*) FROM trusted_scan_events) AS trusted_rows,
  (SELECT COUNT(*) FROM quarantine_scan_events) AS quarantine_rows,
  (SELECT COUNT(*) FROM candidate_scan_events) - (SELECT COUNT(*) FROM trusted_scan_events) - (SELECT COUNT(*) FROM quarantine_scan_events) AS difference,
  CASE WHEN (SELECT COUNT(*) FROM candidate_scan_events) = (SELECT COUNT(*) FROM trusted_scan_events) + (SELECT COUNT(*) FROM quarantine_scan_events)
    THEN 'PASS' ELSE 'FAIL' END AS reconciliation_result,
  (SELECT COUNT(*) FROM trusted_scan_events t INNER JOIN quarantine_scan_events q ON t.record_id = q.record_id) AS overlap_count;

candidate_rows,trusted_rows,quarantine_rows,difference,reconciliation_result,overlap_count
710666,710666,0,0,PASS,0


In [ ]:
%sql
-- ============================================================================
-- ENTITY 5: EXCEPTIONS (silver_exceptions — 18352 rows)
-- Rules: DQ-EXC-001 (identity), DQ-EXC-002 (reference), DQ-EXC-003 (schema),
--        DQ-EXC-004 (chronology)
-- ============================================================================

CREATE OR REPLACE TABLE candidate_exceptions AS SELECT * FROM silver_exceptions;

CREATE OR REPLACE TABLE dq_exceptions_eval AS
SELECT
  c.*,
  COUNT(*) OVER (PARTITION BY c.exception_id) AS exc_id_dup_count,
  s.shipment_id AS ref_shipment_id,
  (c.source_record_id IS NULL OR c.exception_id IS NULL
   OR COUNT(*) OVER (PARTITION BY c.exception_id) > 1) AS fail_dq_exc_001,
  (c.shipment_id IS NOT NULL AND s.shipment_id IS NULL) AS fail_dq_exc_002,
  (c.exception_type NOT IN ('ADDRESS_ISSUE','PACKAGE_DAMAGED','MISSED_CONNECTION','OTHER','WEATHER','CAPACITY','CUSTOMS')
   OR c.severity NOT IN ('LOW','MEDIUM','HIGH')
   OR c.resolution_status NOT IN ('OPEN','RESOLVED','CLOSED')) AS fail_dq_exc_003,
  (c.resolution_time IS NOT NULL AND c.exception_time IS NOT NULL
   AND c.resolution_time < c.exception_time) AS fail_dq_exc_004
FROM candidate_exceptions c
LEFT JOIN (SELECT DISTINCT shipment_id FROM silver_shipments) s ON c.shipment_id = s.shipment_id;

CREATE OR REPLACE TABLE trusted_exceptions AS
SELECT source_record_id, exception_id, shipment_id, scan_id, exception_type,
  exception_time, hub_id, severity, resolution_status, resolution_time,
  source_system, run_id, ingestion_timestamp, source_file_name, batch_id
FROM dq_exceptions_eval
WHERE NOT (fail_dq_exc_001 OR fail_dq_exc_002 OR fail_dq_exc_003 OR fail_dq_exc_004);

CREATE OR REPLACE TABLE quarantine_exceptions AS
SELECT source_record_id, exception_id, shipment_id, exception_type,
  ARRAY_COMPACT(ARRAY(
    IF(fail_dq_exc_001, 'DQ-EXC-001', NULL),
    IF(fail_dq_exc_002, 'DQ-EXC-002', NULL),
    IF(fail_dq_exc_003, 'DQ-EXC-003', NULL),
    IF(fail_dq_exc_004, 'DQ-EXC-004', NULL)
  )) AS failed_rule_ids,
  CONCAT_WS(' | ',
    IF(fail_dq_exc_001, 'DQ-EXC-001: identity/null/duplicate exception_id', NULL),
    IF(fail_dq_exc_002, 'DQ-EXC-002: referenced shipment does not exist', NULL),
    IF(fail_dq_exc_003, 'DQ-EXC-003: invalid enum value', NULL),
    IF(fail_dq_exc_004, 'DQ-EXC-004: resolution before exception time', NULL)
  ) AS failure_details
FROM dq_exceptions_eval
WHERE fail_dq_exc_001 OR fail_dq_exc_002 OR fail_dq_exc_003 OR fail_dq_exc_004;

SELECT
  (SELECT COUNT(*) FROM candidate_exceptions) AS candidate_rows,
  (SELECT COUNT(*) FROM trusted_exceptions) AS trusted_rows,
  (SELECT COUNT(*) FROM quarantine_exceptions) AS quarantine_rows,
  (SELECT COUNT(*) FROM candidate_exceptions) - (SELECT COUNT(*) FROM trusted_exceptions) - (SELECT COUNT(*) FROM quarantine_exceptions) AS difference,
  CASE WHEN (SELECT COUNT(*) FROM candidate_exceptions) = (SELECT COUNT(*) FROM trusted_exceptions) + (SELECT COUNT(*) FROM quarantine_exceptions)
    THEN 'PASS' ELSE 'FAIL' END AS reconciliation_result,
  (SELECT COUNT(*) FROM trusted_exceptions t INNER JOIN quarantine_exceptions q ON t.source_record_id = q.source_record_id) AS overlap_count;

candidate_rows,trusted_rows,quarantine_rows,difference,reconciliation_result,overlap_count
18352,18330,24,-2,FAIL,0


In [ ]:
%sql
-- ============================================================================
-- CROSS-ENTITY DQ SUMMARY
-- Failed counts per rule across all 5 entities
-- ============================================================================

SELECT 'HUBS' AS entity, 'DQ-HUB-001' AS rule_id, SUM(CASE WHEN fail_dq_hub_001 THEN 1 ELSE 0 END) AS failed_count FROM dq_hubs_eval
UNION ALL SELECT 'HUBS', 'DQ-HUB-002', SUM(CASE WHEN fail_dq_hub_002 THEN 1 ELSE 0 END) FROM dq_hubs_eval
UNION ALL SELECT 'HUBS', 'DQ-HUB-003', SUM(CASE WHEN fail_dq_hub_003 THEN 1 ELSE 0 END) FROM dq_hubs_eval
UNION ALL SELECT 'CARRIERS', 'DQ-CAR-001', SUM(CASE WHEN fail_dq_car_001 THEN 1 ELSE 0 END) FROM dq_carriers_eval
UNION ALL SELECT 'CARRIERS', 'DQ-CAR-002', SUM(CASE WHEN fail_dq_car_002 THEN 1 ELSE 0 END) FROM dq_carriers_eval
UNION ALL SELECT 'CARRIERS', 'DQ-CAR-003', SUM(CASE WHEN fail_dq_car_003 THEN 1 ELSE 0 END) FROM dq_carriers_eval
UNION ALL SELECT 'ROUTES', 'DQ-RT-001', SUM(CASE WHEN fail_dq_rt_001 THEN 1 ELSE 0 END) FROM dq_routes_eval
UNION ALL SELECT 'ROUTES', 'DQ-RT-002', SUM(CASE WHEN fail_dq_rt_002 THEN 1 ELSE 0 END) FROM dq_routes_eval
UNION ALL SELECT 'ROUTES', 'DQ-RT-003', SUM(CASE WHEN fail_dq_rt_003 THEN 1 ELSE 0 END) FROM dq_routes_eval
UNION ALL SELECT 'ROUTES', 'DQ-RT-004', SUM(CASE WHEN fail_dq_rt_004 THEN 1 ELSE 0 END) FROM dq_routes_eval
UNION ALL SELECT 'ROUTES', 'DQ-RT-005', SUM(CASE WHEN fail_dq_rt_005 THEN 1 ELSE 0 END) FROM dq_routes_eval
UNION ALL SELECT 'SCAN_EVENTS', 'DQ-SCN-001', SUM(CASE WHEN fail_dq_scn_001 THEN 1 ELSE 0 END) FROM dq_scan_events_eval
UNION ALL SELECT 'SCAN_EVENTS', 'DQ-SCN-002', SUM(CASE WHEN fail_dq_scn_002 THEN 1 ELSE 0 END) FROM dq_scan_events_eval
UNION ALL SELECT 'SCAN_EVENTS', 'DQ-SCN-003', SUM(CASE WHEN fail_dq_scn_003 THEN 1 ELSE 0 END) FROM dq_scan_events_eval
UNION ALL SELECT 'EXCEPTIONS', 'DQ-EXC-001', SUM(CASE WHEN fail_dq_exc_001 THEN 1 ELSE 0 END) FROM dq_exceptions_eval
UNION ALL SELECT 'EXCEPTIONS', 'DQ-EXC-002', SUM(CASE WHEN fail_dq_exc_002 THEN 1 ELSE 0 END) FROM dq_exceptions_eval
UNION ALL SELECT 'EXCEPTIONS', 'DQ-EXC-003', SUM(CASE WHEN fail_dq_exc_003 THEN 1 ELSE 0 END) FROM dq_exceptions_eval
UNION ALL SELECT 'EXCEPTIONS', 'DQ-EXC-004', SUM(CASE WHEN fail_dq_exc_004 THEN 1 ELSE 0 END) FROM dq_exceptions_eval
ORDER BY entity, rule_id;

## Rule Applicability Matrix

The 8 shipment-specific DQ rules (DQ-SHP-001 through DQ-EVT-001) apply to the shipments entity only.
Each of the other 5 entities has its own entity-specific rule set documented below.

| Entity | Shipment Rules (DQ-SHP…EVT) | Entity-Specific Rules |
|--------|----------------------------|----------------------|
| **Shipments** | DQ-SHP-001, DQ-REF-001, DQ-TIM-001, DQ-SCN-001, DQ-RTE-001, DQ-DEL-001, DQ-MEA-001, DQ-EVT-001 (all 8 FULL) | — |
| **Hubs** | N/A — hubs have no shipment_id, timestamps, measures, or delivery status | DQ-HUB-001 (identity), DQ-HUB-002 (schema/enum), DQ-HUB-003 (date chronology) |
| **Carriers** | N/A — carriers have no shipment_id, scan sequences, or delivery status | DQ-CAR-001 (identity), DQ-CAR-002 (schema/enum), DQ-CAR-003 (date chronology) |
| **Routes** | N/A — routes are a reference table, not a transactional entity | DQ-RT-001 (identity), DQ-RT-002 (reference), DQ-RT-003 (schema), DQ-RT-004 (dates), DQ-RT-005 (measures) |
| **Scan Events** | N/A — scan events have no delivery status or route consistency | DQ-SCN-001 (identity), DQ-SCN-002 (schema/enum), DQ-SCN-003 (timestamp validity) |
| **Exceptions** | N/A — exceptions have no shipment identity or route consistency | DQ-EXC-001 (identity), DQ-EXC-002 (reference), DQ-EXC-003 (schema/enum), DQ-EXC-004 (chronology) |

### Why shipment rules do not apply to other entities

- **DQ-SHP-001** (shipment identity): Only shipments have shipment_id and order_reference.
- **DQ-REF-001** (reference integrity): Entity-specific reference checks are defined per entity (e.g., DQ-RT-002 for routes, DQ-EXC-002 for exceptions).
- **DQ-TIM-001** (chronology): Only shipments have the full timestamp chain (booking → pickup → delivery → return). Other entities have simpler date checks.
- **DQ-SCN-001** (scan sequence): Only scan events have event_sequence_no. Scan events have their own identity rule.
- **DQ-RTE-001** (route consistency): Only shipments reference routes. Routes are themselves the reference table.
- **DQ-DEL-001** (delivery/status): Only shipments have delivery_outcome and latest_status.
- **DQ-MEA-001** (measures): Only shipments have package_weight_kg, package_count, freight_amount. Routes have distance_km and expected_transit_hours (DQ-RT-005).
- **DQ-EVT-001** (event/schema): Each entity has its own schema validation rule (DQ-HUB-002, DQ-CAR-002, DQ-RT-003, DQ-SCN-002, DQ-EXC-003).